# Day 3 · Polygons — ประเทศไทยถูกตัดเป็นหกเหลี่ยม 1,738 บล็อก

**#30DayMapChallenge 2026 · วันที่ 3 พ.ย. · ธีม Polygons**

ต่อจาก Day 1 (จุดดูดาว 83 แห่ง) และ Day 2 (เส้นเส้นทางขับรถ) — วันนี้ถามคำถามที่
จุดกับเส้นตอบไม่ได้: **"พื้นที่ของไทยกี่ตารางกิโลเมตรที่ใกล้พอจะดูดาวยัง"**

สิ่งที่ต้องใช้คือ **polygon** ล้วน ๆ: ตัดประเทศเป็นตารางหกเหลี่ยม (H3) แล้ว ระบายสีแต่ละบล็อก
ตามระยะทางถึงจุดฟ้ามืดที่ใกล้ที่สุด

**Mapbox เป็นสปอนเซอร์ของชาเลนจ์ปีนี้** — โน้ตบุ๊กนี้จึงใช้ polygon ขอบประเทศ
จาก tileset ฟรีของ Mapbox เป็นแหล่งหลัก (admin-0 `country-boundaries-v1`)
ถ้าไม่มี token จะสลับไปใช้ GADM ให้อัตโนมัติ แล้วเปรียบเทียบความต่างให้ดูท้ายโน้ตบุ๊ก

> รันตามได้เลย: กด **Runtime → Run all** — ขอบประเทศต้องเลือกเองว่าใช้ Mapbox หรือ GADM (ดูเซลล์ถัดไป)

In [ ]:
# ติดตั้ง library ที่ Colab ไม่มีมาให้ (numpy/pandas/matplotlib มีอยู่แล้ว)
!pip install -q h3 shapely folium fontTools
import h3, shapely, folium
print("h3", h3.__version__, "| shapely", shapely.__version__, "| folium", folium.__version__)

## 0. เตรียมฟอนต์ไทย

ใช้ **Noto Sans Thai** ตัวเดียวกับ Day 1 — เพราะทดสอบแล้วว่าวาดสระบน/วรรณยุกต์ที่ซ้อนกัน
(เช่น "ท้องฟ้า") ได้ถูกต้อง ส่วน Sarabun วาดสระซ้อนไม่ชัด

In [ ]:
# ติดตั้งฟอนต์ภาษาไทย Noto Sans Thai (รันเซลล์นี้ก่อนเซลล์วาดภาพเสมอ)
import urllib.request
from fontTools.ttLib import TTFont
from fontTools.varLib.instancer import instantiateVariableFont
from matplotlib import font_manager

urllib.request.urlretrieve(
    "https://github.com/google/fonts/raw/main/ofl/notosansthai/"
    "NotoSansThai%5Bwdth%2Cwght%5D.ttf",
    "NotoSansThai-VF.ttf",
)
for _name, _wght in [("NotoSansThai-Regular", 400), ("NotoSansThai-Bold", 700)]:
    _f = TTFont("NotoSansThai-VF.ttf")
    instantiateVariableFont(_f, {"wght": _wght, "wdth": 100}, inplace=True)
    _f.save(f"{_name}.ttf")
    font_manager.fontManager.addfont(f"{_name}.ttf")

import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Noto Sans Thai"
print("ติดตั้งฟอนต์ไทย Noto Sans Thai เรียบร้อย")

## 1. Token ของ Mapbox — เก็บใน Secrets อย่า hardcode

Mapbox Boundaries มี 2 ระดับที่ใช้ฟรีไม่เท่ากัน (เช็คจาก docs ณ วันที่ทำ):

| ระดับ | tileset | ฟรีไหม |
|---|---|---|
| admin-0 (ประเทศ) | `mapbox.country-boundaries-v1` | ✅ ฟรีให้ผู้ใช้ Mapbox ทุกคน |
| admin-1+ (จังหวัด/อำเภอ) | Mapbox Boundaries v4 | ❌ เสียเงิน ต้องติดต่อ sales |

และ docs ระบุชัดว่า tileset **"Raw geometries are not provided for offline use"**
— วาดบน Mapbox GL ได้ แต่จะดูด geometry ดิบไปใช้เองต้องซื้อ license แบบ GeoJSON flat file

**ทางรอดของงานนี้:** ใช้ **Tilequery API** which คืน geometry เป็น **GeoJSON**
เรายิงถามว่า "จุดนี้เป็นของประเทศอะไร" แล้วเอา polygon ของไทยกลับมาใช้ได้ฟรี

อย่าใส่ token ในโค้ด — เก็บใน **Colab Secrets** (ไอคอนรูปกุญแจแถบซ้าย) ชื่อ `MAPBOX_TOKEN`
แล้วดึงด้วย `userdata.get()` โน้ตบุ๊กนี้แชร์ต่อได้โดย token ไม่หลุด

In [ ]:
MAPBOX_TOKEN = None
try:
    from google.colab import userdata
    MAPBOX_TOKEN = userdata.get("MAPBOX_TOKEN")
    print("พบ MAPBOX_TOKEN ใน Colab Secrets → ใช้ขอบประเทศจาก Mapbox")
except Exception:
    print("ไม่พบ MAPBOX_TOKEN ใน Colab Secrets → สลับไปใช้ GADM v4.1 (ฟรี ไม่ต้องมี token)")

## 2. ข้อมูลจุดดูดาว 83 แห่ง (ต่อจาก Day 1)

Day 1 ดึงมาจาก REST API ของ NARIT (`darksky.narit.or.th/wp-json/wp/v2/darksky`)
เก็บเป็น CSV ไว้ใน repo — โหลดจาก GitHub ได้เลย ถ้าไม่ได้ให้อัปโหลดไฟล์เอง

In [ ]:
import pandas as pd
CSV_URL = "https://raw.githubusercontent.com/thammarat-ai/geo-notes/main/data/darksky_thailand.csv"
try:
    pts = pd.read_csv(CSV_URL)
    print("โหลด CSV จาก GitHub ได้")
except Exception:
    from google.colab import files
    up = files.upload()   # กด Choose Files แล้วเลือก darksky_thailand.csv
    pts = pd.read_csv(next(iter(up.values())))

print(len(pts), "จุดดูดาว |", pts.province.nunique(), "จังหวัด")
pts[["name", "province", "latitude", "longitude"]].head(3)

## 3. ขอบประเทศไทยเป็น polygon

### ทางที่ 1: Mapbox Tilequery (ฟรี · high resolution)

Tilequery คืน feature ที่ทับจุดที่ถาม — ผมยิง 5 จุดกระจายทั่วไทย
(ภาคกลาง เหนือ อีสาน ตะวันออก ใต้) เพื่อเก็บ polygon ของ `TH` ให้ครบทุกเกาะ
แล้วเอามาต่อกันเป็น MultiPolygon

ข้อควรรู้: geometry จาก tileset ถูก **clip ตามขอบ tile** (มี buffer 4 พิกเซล)
จุดที่ใกล้ขอบ tileอาจได้ polygon ไม่ครบทั้งประเทศ — นี่คือเหตุผลที่ยิงหลายจุดแล้ว union กัน

In [ ]:
import json, math, requests
from urllib.request import urlopen

def fetch_mapbox(token):
    seeds = [(100.5, 13.7), (99.0, 16.0), (100.1, 18.5), (102.1, 15.5), (98.5, 8.0)]
    feats = {}
    for lon, lat in seeds:
        r = requests.get(
            "https://api.mapbox.com/v4/mapbox.country-boundaries-v1"
            f"/tilequery/{lon},{lat}.json",
            params={"access_token": token, "limit": 10, "layers": "country_boundaries"},
            timeout=60)
        r.raise_for_status()
        for ft in r.json().get("features", []):
            p = ft["properties"]
            if p.get("iso_3166_1") == "TH" and p.get("disputed") == "false":
                feats[p["mapbox_id"]] = ft["geometry"]
    if not feats:
        raise RuntimeError("ไม่พบ polygon ของ TH จาก Tilequery")
    geoms = list(feats.values())
    geom = geoms[0] if len(geoms) == 1 else {
        "type": "MultiPolygon",
        "coordinates": [g["coordinates"] if g["type"] == "Polygon" else g["coordinates"]
                        for g in geoms]}
    n_vtx = sum(len(r) for poly in geom["coordinates"]
                for r in (poly if geom["type"] == "MultiPolygon" else [poly]))
    return geom, f"Mapbox country-boundaries-v1 ({len(geoms)} ชิ้น, {n_vtx:,} จุดยอด)"

def fetch_gadm():
    with urlopen("https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_THA_0.json",
                 timeout=120) as r:
        gj = json.load(r)
    geom = gj["features"][0]["geometry"]
    n_vtx = sum(len(ring) for poly in geom["coordinates"] for ring in poly)
    return geom, f"GADM v4.1 ({len(geom['coordinates'])} ส่วน, {n_vtx:,} จุดยอด)"

if MAPBOX_TOKEN:
    try:
        geom, MASK_SRC = fetch_mapbox(MAPBOX_TOKEN)
    except Exception as e:
        print("Mapbox ล้มเหลว:", type(e).__name__, str(e)[:120], "→ ใช้ GADM แทน")
        geom, MASK_SRC = fetch_gadm()
else:
    geom, MASK_SRC = fetch_gadm()

print("ขอบประเทศที่ใช้:", MASK_SRC)
print("geometry type:", geom["type"])

### คำนวณพื้นที่ polygon ด้วยทรงกลม (ไม่ต้องพึ่ง pyproj)

อยากเทียบกับตัวเลขทางการ (ไทย ≈ 513,120 ตร.กม.) ว่า mask ที่เราได้แม่นแค่ไหน
ใช้สูตร **spherical polygon area** (Chamberlain & Duquette) บน lon/lat ตรง ๆ ได้เลย

In [ ]:
def ring_area_km2(ring):
    tot = 0.0
    R = 6371.0088
    for i in range(len(ring) - 1):
        lon1, lat1 = math.radians(ring[i][0]), math.radians(ring[i][1])
        lon2, lat2 = math.radians(ring[i + 1][0]), math.radians(ring[i + 1][1])
        tot += (lon2 - lon1) * (2 + math.sin(lat1) + math.sin(lat2))
    return abs(tot * R * R / 2.0)

parts = geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]
rings, areas = [], []
for poly in parts:
    outer = [tuple(c[:2]) for c in poly[0]]
    holes = [[tuple(c[:2]) for c in h] for h in poly[1:]]
    a = ring_area_km2(outer) - sum(ring_area_km2(h) for h in holes)
    if a < 300:          # หกเหลี่ยม res 5 ใหญ่ ~289 ตร.กม. -> เกาะเล็กกว่านั้นใส่บล็อกเดียวไม่ได้
        continue
    rings.append((outer, holes))
    areas.append(a)

print(f"เก็บ {len(rings)} จาก {len(parts)} ส่วน (ตัดเกาะจิ๋วทิ้ง)")
print(f"พื้นที่ mask = {sum(areas):,.0f} ตร.กม.  (อ้างอิงทางการ ~513,120)")

## 4. ขั้นที่ 1 ของงาน polygon: เต็มประเทศด้วยหกเหลี่ยม H3

**ทำไม H3 แทน grid สี่เหลี่ยม?** — เพราะสี่เหลี่ยมมีเพื่อนบ้าน 2 แบบ
(ติดด้าน = ไกล 1 หน่วย, ติดมุม = ไกล 1.41 หน่วย) ทำให้ระยะทางเพี้ยน
หกเหลี่ยมมีเพื่อนบ้าน 6 ตัวห่างเท่ากันหมด เหมาะกับงาน "ใกล้/ไกล" มาก

`polygon_to_cells` รับ polygon แล้วคืนทุก cell ที่อยู่ด้านใน — นี่คือหัวใจของวันนี้
เราแปลง "ประเทศ" (polygon) เป็น "ตารางของ polygon เล็ก ๆ" ที่นับพื้นที่ได้

In [ ]:
import h3
RES = 5

# หมายเหตุลำดับพิกัด: GeoJSON เก็บ (lon, lat) แต่ H3 รับ (lat, lon) — สลับผิดคือแผนที่หลุดโลก
polys = [h3.LatLngPoly([(la, lo) for lo, la in o],
                        *[[(la, lo) for lo, la in h] for h in hs])
         for o, hs in rings]

cells = set()
for p in polys:
    cells |= set(h3.polygon_to_cells(p, RES))
cells = sorted(cells)

one = h3.cell_area(cells[0], unit="km^2")
print(f"H3 resolution {RES}: {len(cells):,} บล็อก")
print(f"บล็อกละ ~{one:,.1f} ตร.กม. | รวม {len(cells)*one:,.0f} ตร.กม.")
print(f"ถ้าใช้ res 6 (ละเอียดขึ้น 7 เท่า) จะได้ ~{len(cells)*7:,} บล็อก — ช้าและไฟล์ใหญ่ขึ้นมาก")

## 5. ขั้นที่ 2: ระยะทางถึงจุดดูดาวที่ใกล้ที่สุดของแต่ละบล็อก

ต่อจาก Day 1 — จุด 83 แห่งคือ "แหล่งฟ้ามืด" ที่ NARIT รับรอง
แต่ละหกเหลี่ยมถามว่าไกลจากจุดไหนที่สุด (เส้นตรง haversine) แล้ว ระบายสี

In [ ]:
import numpy as np
PLNG, PLAT = pts.longitude.to_numpy(), pts.latitude.to_numpy()

def hav_km(la1, lo1, la2s, los):
    p1 = math.radians(la1); l1 = math.radians(lo1)
    p2 = np.radians(la2s); l2 = np.radians(los)
    dp = p2 - p1; dl = l2 - l1
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * 6371.0088 * np.arcsin(np.sqrt(a))

rows = []
for c in cells:
    la, lo = h3.cell_to_latlng(c)
    d = hav_km(la, lo, PLAT, PLNG)
    j = int(np.argmin(d))
    rows.append({"cell": c, "lat": la, "lon": lo, "km": float(d.min()),
                 "nearest": pts.name[j], "province": pts.province[j],
                 "area_km2": h3.cell_area(c, unit="km^2")})
H = pd.DataFrame(rows)

BINS = [(25, "#22d3ee", "< 25 กม."), (50, "#4ade80", "25–50 กม."),
        (100, "#facc15", "50–100 กม."), (150, "#fb923c", "100–150 กม."),
        (float("inf"), "#f43f5e", "> 150 กม.")]

def bucket(k):
    for lim, col, lab in BINS:
        if k < lim: return lab, col

H["bucket"], H["color"] = zip(*H.km.map(bucket))
tot_area = H.area_km2.sum()

print(f"(percentile) p5={np.percentile(H.km,5):.0f} p50={H.km.median():.0f} "
      f"p95={np.percentile(H.km,95):.0f} max={H.km.max():.0f} กม.")
for _, _, lab in BINS:
    s = H[H.bucket == lab]
    print(f"  {lab:<11} {len(s):>5} บล็อก {s.area_km2.sum():>9,.0f} ตร.กม."
          f"  {s.area_km2.sum()/tot_area*100:>5.1f}%")
far = H.loc[H.km.idxmax()]
print(f"บล็อกไกลสุด: {far.km:.0f} กม. จาก {far.nearest} ({far.province})")

## 6. ขั้นที่ 3: **ตัดหกเหลี่ยมที่ล้นออกนอกประเทศ** (polygon clipping)

นี่คือขั้นที่ทำให้ "polygons" เป็น polygons จริง — หกเหลี่ยมตรงชายแดนและตามเกาะ
ล้นออกทะเล/ออกประเทศเพื่อนบ้าน ถ้าไม่ตัด แผนที่จะไม่เหลือรูปทรงไทยเลย

ใช้ **shapely** หาผลตัด (intersection) ระหว่างหกเหลี่ยมกับ mask หลายชิ้น
แล้วเก็บทุก ring ที่เหลือ (หกเหลี่ยมบางบล็อกถูกตัดเป็น 2 ชิ้นเมื่อข้ามเกาะ/อ่าว)

> ข้อจำกัดที่ควรรู้: shapely ทำงานบนระนาบ (lon/lat เป็นตัวเลขดิบ) ไม่ใช่ทรงกลม
> ที่ละติจูดไทย ความคลาดเคลื่อนของพื้นที่จากการทำ planar นี้ < ~1% รับได้กับงาน visualization

In [ ]:
from shapely.geometry import Polygon as SPoly, MultiPolygon
from shapely.ops import unary_union

MASK = MultiPolygon([SPoly(o, h) for o, h in rings])

def rr(ring):  # 5 ทศนิยม ~= 1 เมตร ลดขนาดไฟล์ได้หลายเท่าโดยไม่เสียรายละเอียดที่ตามองเห็น
    return [(round(float(x), 5), round(float(y), 5)) for x, y in ring]

clipped, n_clip = {}, 0
for c in H.cell:
    g = SPoly([(lo, la) for la, lo in h3.cell_to_boundary(c)])
    if not g.within(MASK):
        inter = unary_union([s.intersection(g) for s in MASK.geoms])
        if inter.is_empty: continue
        n_clip += 1
        clipped[c] = ([rr(inter.exterior.coords)] if inter.geom_type == "Polygon"
                      else [rr(p.exterior.coords) for p in inter.geoms])
    else:
        clipped[c] = [rr(g.exterior.coords)]

print(f"ถูกตัดขอบ: {n_clip:,} จาก {len(H):,} บล็อก ({n_clip/len(H)*100:.1f}%)")
print("นี่คือเหตุผลที่ polygon clipping สำคัญ — 14% ของภาพมาจากบล็อกที่แตะขอบประเทศ")

## 7. วาดแผนที่ธีมท้องฟ้ายามค่ำคืน (ต่อธีมจาก Day 1)

แต่ละ ring ของหกเหลี่ยมกลายเป็น `matplotlib.patches.Polygon` — ไม่ต้องใช้ geopandas
พื้นหลังเข้ม + ดาวสุ่ม + จุดดูดาวเรืองแสง 2 ชั้น (halo + core) เหมือน Day 1

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon

BG, PANEL = "#070b18", "#101a3a"
fig = plt.figure(figsize=(10.5, 12.5), dpi=200)
ax = fig.add_axes([0.02, 0.055, 0.96, 0.86])
fig.patch.set_facecolor(BG); ax.set_facecolor(BG)

rng = np.random.default_rng(7)
ax.scatter(rng.uniform(97.4, 105.9, 500), rng.uniform(5.4, 20.7, 500),
           s=0.6, c="white", alpha=0.35, zorder=0)

cell_rgba = dict(zip(H.cell, H.color))
for c, coords in clipped.items():
    for ring in coords:
        ax.add_patch(MplPolygon(np.asarray(ring), closed=True, facecolor=cell_rgba[c],
                                edgecolor="#0b1226", lw=0.25, alpha=0.92, zorder=3))
for ring, hs in rings:
    ax.add_patch(MplPolygon(np.asarray(ring), closed=True, fill=False,
                            edgecolor="#8ea2ff", lw=0.8, zorder=5))
ax.scatter(PLNG, PLAT, s=210, c="white", alpha=0.14, linewidths=0, zorder=6)
ax.scatter(PLNG, PLAT, s=34, c="white", edgecolors="#8ea2ff", linewidths=0.5, zorder=7)

ax.set_xlim(97.4, 105.9); ax.set_ylim(5.4, 20.7)
ax.set_aspect(1.6); ax.axis("off")

handles = [plt.Rectangle((0, 0), 1, 1, color=col) for _, col, lab in BINS]
leg = ax.legend(handles, [lab for _, _, lab in BINS], loc="lower right", frameon=True,
                facecolor=PANEL, edgecolor="#2b3a6b", labelcolor="white", fontsize=9.5,
                borderpad=0.9, labelspacing=0.8, title="ระยะถึงจุดฟ้ามืดที่ใกล้ที่สุด (km)")
leg.get_frame().set_alpha(0.9); leg.get_title().set_color("#cdd6ff")

fig.text(0.035, 0.965, f"Day 3 · Polygons — ประเทศไทยถูกตัดเป็นหกเหลี่ยม {len(H):,} บล็อก",
         color="white", fontsize=17, fontweight="bold", va="top")
fig.text(0.035, 0.929, f"ตาราง H3 ระดับที่ {RES} · สี = ระยะทางจากจุดอนุรักษ์ท้องฟ้ามืด 83 แห่งของ NARIT",
         color="#9fb0e8", fontsize=10.5, va="top")
fig.text(0.035, 0.028, f"ขอบประเทศ: {MASK_SRC} · พื้นที่รวม {tot_area:,.0f} ตร.กม.",
         color="#5f6f9e", fontsize=8)
fig.text(0.965, 0.028, "#30DayMapChallenge · Geo Notes", color="#5f6f9e", fontsize=8, ha="right")

fig.savefig("day03_polygons_preview.png", dpi=200, facecolor=BG)
plt.show()

## 8. ส่งออก polygon ให้เอาไปใช้ต่อ

**GeoJSON** เปิดใน QGIS ได้ทันที — ทุกบล็อกมี properties `km_nearest` / `bucket` / `nearest`
เอาไป join กับข้อมูลอย่างอื่นต่อได้ (นี่คือข้อได้เปรียบของ polygon: เป็น *ข้อมูล* ไม่ใช่แค่ภาพ)

In [ ]:
def geom_of(coords):
    # ring list -> GeoJSON geometry (Polygon, หรือ MultiPolygon เมื่อหกเหลี่ยมถูกตัดเป็นชิ้น)
    if len(coords) == 1:
        return {"type": "Polygon", "coordinates": coords}
    return {"type": "MultiPolygon", "coordinates": [[r] for r in coords]}

idx = H.set_index("cell")
gj = {"type": "FeatureCollection", "features": []}
for c, coords in clipped.items():
    r = idx.loc[c]
    gj["features"].append({"type": "Feature", "geometry": geom_of(coords),
        "properties": {"cell": c, "km_nearest": round(r.km, 1), "bucket": r.bucket,
                       "nearest": r.nearest, "province": r.province,
                       "area_km2": round(r.area_km2, 1)}})
with open("day03_hex_polygons.geojson", "w", encoding="utf-8") as f:
    json.dump(gj, f, ensure_ascii=False)

import os
print("GeoJSON:", round(os.path.getsize("day03_hex_polygons.geojson")/1024, 1), "KB",
      "|", len(gj["features"]), "features")

## 9. แผนที่ interactive (folium)

แบ่ง layer ตามช่วงระยะ เปิด–ปิดทีละช่วงได้ — พื้นหลังเป็น Mapbox Dark ถ้ามี token
ถ้าไม่มีใช้ CARTO dark (ฟรี ไม่ต้องมี token)

In [ ]:
import folium
m = folium.Map(location=[13.1, 100.5], zoom_start=6, tiles=None)
if MAPBOX_TOKEN:
    folium.TileLayer(
        tiles=f"https://api.mapbox.com/styles/v1/mapbox/dark-v11/tiles/{{z}}/{{x}}/{{y}}"
              f"?access_token={MAPBOX_TOKEN}",
        attr="© Mapbox © OpenStreetMap", name="Mapbox Dark", max_zoom=20).add_to(m)
else:
    folium.TileLayer(
        tiles="https://cartodb-basemaps-a.global.ssl.fastly.net/dark_all/{z}/{x}/{y}.png",
        attr="© OpenStreetMap © CARTO", name="Dark", max_zoom=19).add_to(m)

for (b, col), grp in H.groupby(["bucket", "color"], sort=False):
    fg = folium.FeatureGroup(name=f"{b} ({len(grp)} บล็อก)").add_to(m)
    folium.GeoJson({"type": "FeatureCollection", "features": [
        {"type": "Feature", "geometry": geom_of(clipped[c]),
         "properties": {"cell": c, "km_nearest": round(idx.loc[c].km, 1),
                        "nearest": idx.loc[c].nearest,
                        "province": idx.loc[c].province}} for c in grp.cell]},
        style_function=lambda x, col=col: {"fillColor": col, "color": col,
                                           "weight": 0.3, "fillOpacity": 0.72},
        tooltip=folium.GeoJsonTooltip(["km_nearest", "nearest", "province"], labels=True)
        ).add_to(fg)

for r in pts.itertuples():
    folium.CircleMarker([r.latitude, r.longitude], radius=2.5, color="white",
                        fill=True, fill_color="white", fill_opacity=0.9,
                        tooltip=str(r.name)).add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.save("day03_polygons_map.html")
from IPython.display import HTML, display
display(HTML(filename="day03_polygons_map.html"))

## 10. เปรียบเทียบขอบประเทศ: Mapbox vs GADM

รันเซลล์นี้เพื่อดูว่า mask คนละแหล่งให้ผลต่างกันแค่ไหน (จำนวนจุดยอด พื้นที่
และจำนวนหกเหลี่ยมที่ได้) — เป็นเหตุผลว่าทำไม Day 1 เลิกใช้ Natural Earth 110m

In [ ]:
report = {"แหล่งที่ใช่วันนี้": MASK_SRC,
          "พื้นที่ mask (ตร.กม.)": round(sum(areas), 0),
          "พื้นที่อ้างอิงทางการ": 513120,
          "จำนวนส่วน (แผ่นดิน+เกาะ) ที่เก็บ": len(rings),
          "จำนวนหกเหลี่ยม res 5": len(cells),
          "ผลต่างจากค่าอ้างอิง": f"{(sum(areas)-513120)/513120*100:+.2f}%"}
for k, v in report.items():
    print(f"{k:<34} {v}")
print()
print("ถ้าสลับ tileset/แหล่งข้อมูลแล้วจำนวนบล็อกเปลี่ยนเกิน ~1% "
      "แปลว่า mask รายละเอียดต่างกันมีผลต่อการวิเคราะห์จริง")

## สรุปสิ่งที่ได้วันนี้

- **Polygon ให้คำตอบที่ Point/Line ให้ไม่ได้** — "พื้นที่กี่ ตร.กม." ต้องมี polygon เป็นหน่วยนับ
- **Mapbox ให้ polygon ประเทศฟรี** (`country-boundaries-v1` + Tilequery) แต่ระดับจังหวัดต้องซื้อ
  ทางเลือกฟรีคือ H3 (สร้างเอง) / Natural Earth / GADM / geoSAPI ของไทย
- ** clipping คือ 14% ของงาน** — บล็อกที่แตะขอบประเทศต้องตัด ไม่งั้นเสียรูปทรง
- ลำดับพิกัด `(lat, lon)` ของ H3 ไม่ตรงกับ `(lon, lat)` ของ GeoJSON — สลับผิดคือแผนที่หลุดโลก

ไฟล์ที่ได้: `day03_polygons_preview.png` · `day03_hex_polygons.geojson` · `day03_polygons_map.html`